# nDFU on POPQUORN / Potato-Prolific annotations

This notebook applies normalized Distance from Unimodality (nDFU) to the [POPQUORN Potato-Prolific dataset](https://github.com/jiaxin-pei/potato-prolific-dataset) by Pei and Jurgens.

The goal is not to claim that nDFU directly measures “polarization.” Instead, we use it as a compact description of whether an item-level annotation distribution departs from a single-peaked shape. That signal can then guide qualitative inspection, subgroup analysis, or K+1-style modeling.


## 1. What we will analyze

POPQUORN contains several annotation tasks. Three of them are ordinal rating tasks and are natural inputs for nDFU:

- **Offensiveness**: 1-5 rating per text.
- **Politeness**: 1-5 rating per text.
- **Question-answering difficulty**: 1-5 rating per question.

The email rewriting task is free-form text generation, so it is not scored with nDFU here. You could analyze it with other measures, but a distance-from-unimodality metric needs an ordered annotation scale.


## 2. Setup

The notebook uses only lightweight Python tools plus the `ndfu` package. If you run this notebook outside the repository, install `nDFU` from GitHub first.


In [ ]:
# Uncomment when running outside a local checkout of this repository.
# %pip install git+https://github.com/ipavlopoulos/ndfu.git


In [ ]:
from collections import Counter
from pathlib import Path
import textwrap

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from ndfu import dfu, pdf

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 7
SCALE_1_TO_5 = [1, 2, 3, 4, 5]


## 3. Load the POPQUORN rating tasks

The cells below read the raw CSVs directly from GitHub. Each row is one annotation by one annotator. We later group rows by `instance_id` so that each item has a list of ratings and demographic attributes.


In [ ]:
BASE_URL = "https://raw.githubusercontent.com/jiaxin-pei/potato-prolific-dataset/main/dataset"

TASKS = {
    "offensiveness": {
        "url": f"{BASE_URL}/offensiveness/raw_data.csv",
        "label_col": "offensiveness",
        "text_cols": ["text"],
        "description": "How offensive is the text?",
    },
    "politeness": {
        "url": f"{BASE_URL}/politeness_rating/raw_data.csv",
        "label_col": "politeness",
        "text_cols": ["text"],
        "description": "How polite is the text?",
    },
    "qa_difficulty": {
        "url": f"{BASE_URL}/question_answering/raw_data.csv",
        "label_col": "difficulty",
        "text_cols": ["question", "groundtruth", "text"],
        "description": "How difficult is the question-answering item?",
    },
}

raw = {}
for task_name, cfg in TASKS.items():
    df = pd.read_csv(cfg["url"])
    df[cfg["label_col"]] = df[cfg["label_col"]].astype(int)
    raw[task_name] = df
    print(f"{task_name:15s} {df.shape[0]:6d} annotations, {df.instance_id.nunique():5d} items")


## 4. Convert annotations to item-level distributions

For each item we collect all ratings, compute a relative-frequency vector over the 1-5 scale, and then compute nDFU. We also keep annotator demographics as lists so we can later ask whether subgroup-specific distributions explain aggregate non-unimodality.


In [ ]:
DEMOGRAPHIC_COLS = ["gender", "race", "age", "occupation", "education"]


def safe_mode(values):
    counts = Counter(values)
    if not counts:
        return np.nan
    return counts.most_common(1)[0][0]


def summarize_task(df, label_col, text_cols, scale=SCALE_1_TO_5):
    rows = []
    for instance_id, group in df.groupby("instance_id", sort=False):
        scores = group[label_col].dropna().astype(int).tolist()
        if not scores:
            continue
        hist = pdf(scores, scale)
        row = {
            "instance_id": instance_id,
            "n_annotations": len(scores),
            "scores": scores,
            "hist": hist,
            "ndfu": dfu(hist),
            "mean_score": float(np.mean(scores)),
            "majority_score": safe_mode(scores),
        }
        for col in text_cols:
            row[col] = group[col].dropna().iloc[0] if group[col].notna().any() else None
        for col in DEMOGRAPHIC_COLS:
            if col in group.columns:
                row[col] = group[col].dropna().tolist()
        rows.append(row)
    return pd.DataFrame(rows)

summaries = {
    task_name: summarize_task(df, cfg["label_col"], cfg["text_cols"])
    for task_name, (df, cfg) in zip(raw.keys(), zip(raw.values(), TASKS.values()))
}

for task_name, summary in summaries.items():
    print(f"{task_name:15s} {summary.shape[0]:5d} item-level distributions")


## 5. Overview: how much non-unimodality appears?

The table below summarizes nDFU per task. A value of 0 means the rating histogram is compatible with a single peak. Larger values indicate stronger departure from unimodality, not necessarily a richer social interpretation by themselves.


In [ ]:
overview = []
for task_name, summary in summaries.items():
    overview.append({
        "task": task_name,
        "items": len(summary),
        "annotations": int(summary.n_annotations.sum()),
        "mean_annotations": summary.n_annotations.mean(),
        "mean_ndfu": summary.ndfu.mean(),
        "median_ndfu": summary.ndfu.median(),
        "pct_ndfu_gt_0": (summary.ndfu > 0).mean(),
        "pct_ndfu_ge_0_5": (summary.ndfu >= 0.5).mean(),
    })

overview = pd.DataFrame(overview)
overview


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
plot_df = pd.concat(
    [summary.assign(task=task_name) for task_name, summary in summaries.items()],
    ignore_index=True,
)
sns.histplot(data=plot_df, x="ndfu", hue="task", bins=20, element="step", stat="density", common_norm=False, ax=ax)
ax.set_title("Distribution of nDFU by POPQUORN task")
ax.set_xlabel("nDFU: normalized distance from unimodality")
sns.despine()


## 6. Inspect the most non-unimodal items

The next helper prints the items with the largest nDFU scores. This is the most useful first pass: nDFU helps us select examples where the annotation distribution deserves inspection.


In [ ]:
def histogram_table(hist, scale=SCALE_1_TO_5):
    return pd.Series(hist, index=scale).rename("relative_frequency")


def show_top_items(task_name, n=5):
    summary = summaries[task_name].sort_values(["ndfu", "n_annotations"], ascending=False).head(n)
    display_cols = ["instance_id", "n_annotations", "mean_score", "majority_score", "ndfu", "scores"]
    available_text_cols = [col for col in TASKS[task_name]["text_cols"] if col in summary.columns]
    display(summary[display_cols + available_text_cols])
    return summary

for task_name in summaries:
    print("=" * 80)
    print(task_name)
    _ = show_top_items(task_name, n=5)


## 7. Visualize selected item distributions

This plot shows the rating histograms for the highest-nDFU items in each task. The visual shape is often more informative than the number alone.


In [ ]:
def plot_top_histograms(task_name, n=6):
    top = summaries[task_name].sort_values("ndfu", ascending=False).head(n).copy()
    long_rows = []
    for _, row in top.iterrows():
        for score, freq in zip(SCALE_1_TO_5, row["hist"]):
            long_rows.append({
                "instance_id": str(row.instance_id),
                "score": score,
                "relative_frequency": freq,
                "ndfu": row.ndfu,
            })
    long_df = pd.DataFrame(long_rows)
    g = sns.catplot(
        data=long_df,
        x="score",
        y="relative_frequency",
        col="instance_id",
        col_wrap=3,
        kind="bar",
        height=2.2,
        aspect=1.2,
        color="#4C78A8",
        sharey=True,
    )
    g.fig.suptitle(f"Highest-nDFU {task_name} items", y=1.03)
    for ax, (_, row) in zip(g.axes.flat, top.iterrows()):
        ax.set_title(f"{row.instance_id}\\nnDFU={row.ndfu:.2f}")
    return g

for task_name in summaries:
    plot_top_histograms(task_name, n=6)


## 8. Check whether demographic subgroups explain aggregate shape

A high aggregate nDFU can happen for many reasons. One useful diagnostic is to split annotations by a demographic variable and recompute nDFU inside each subgroup.

The function below searches for examples where:

- the aggregate distribution has high nDFU,
- at least two demographic groups have enough annotations,
- subgroup means differ noticeably.

This does not prove causality. It only surfaces examples worth inspecting.


In [ ]:
def subgroup_summary(row, demographic_col, score_col="scores", min_group_size=2):
    scores = row[score_col]
    groups = row.get(demographic_col, [])
    if not isinstance(groups, list) or len(groups) != len(scores):
        return []

    out = []
    for group_name, idxs in pd.Series(groups).groupby(pd.Series(groups)).groups.items():
        group_scores = [scores[i] for i in idxs]
        if len(group_scores) < min_group_size:
            continue
        hist = pdf(group_scores, SCALE_1_TO_5)
        out.append({
            "group": group_name,
            "n": len(group_scores),
            "mean": float(np.mean(group_scores)),
            "ndfu": dfu(hist),
            "hist": hist,
            "scores": group_scores,
        })
    return out


def find_subgroup_contrasts(task_name, demographic_col="gender", min_group_size=2, min_mean_gap=1.0, min_item_ndfu=0.3):
    rows = []
    for _, row in summaries[task_name].iterrows():
        if row.ndfu < min_item_ndfu:
            continue
        groups = subgroup_summary(row, demographic_col, min_group_size=min_group_size)
        if len(groups) < 2:
            continue
        means = [g["mean"] for g in groups]
        gap = max(means) - min(means)
        if gap < min_mean_gap:
            continue
        rows.append({
            "task": task_name,
            "instance_id": row.instance_id,
            "item_ndfu": row.ndfu,
            "mean_gap": gap,
            "groups": groups,
            "text": row.get("text") or row.get("question"),
        })
    return pd.DataFrame(rows).sort_values(["mean_gap", "item_ndfu"], ascending=False)

contrasts = {
    (task_name, demo): find_subgroup_contrasts(task_name, demo)
    for task_name in summaries
    for demo in ["gender", "race"]
}

pd.DataFrame([
    {"task": task, "demographic": demo, "candidate_items": len(df)}
    for (task, demo), df in contrasts.items()
])


## 9. Inspect subgroup candidates

The next cell prints a compact view of examples where subgroup means differ. Start here for qualitative analysis, then return to the raw data for context.


In [ ]:
def describe_contrast(task_name, demographic_col="gender", rank=0):
    df = contrasts[(task_name, demographic_col)]
    if df.empty:
        print(f"No candidates for {task_name} / {demographic_col}")
        return None
    row = df.iloc[rank]
    print(f"Task: {task_name}")
    print(f"Instance: {row.instance_id}")
    print(f"Item nDFU: {row.item_ndfu:.3f}; subgroup mean gap: {row.mean_gap:.3f}")
    print()
    print(textwrap.fill(str(row.text), width=100))
    print()
    for group in row.groups:
        print(f"{group['group']}: n={group['n']}, mean={group['mean']:.2f}, nDFU={group['ndfu']:.2f}, scores={group['scores']}")
    return row

for task_name in summaries:
    print("=" * 80)
    describe_contrast(task_name, "gender", rank=0)


## 10. Prepare a K+1 target

A common use of nDFU is to create a K+1 target: keep the majority class for low-nDFU examples, and assign high-nDFU examples to an additional class. This section only constructs the labels; a downstream classifier can then use them as targets.


In [ ]:
def make_k_plus_one_target(summary, threshold=0.5, extra_label="non_unimodal"):
    target = summary.majority_score.astype(str).copy()
    target = target.where(summary.ndfu < threshold, extra_label)
    return target

kplus_tables = []
for task_name, summary in summaries.items():
    for threshold in [0.25, 0.5, 0.75]:
        target = make_k_plus_one_target(summary, threshold=threshold)
        counts = target.value_counts().rename_axis("label").reset_index(name="count")
        counts.insert(0, "threshold", threshold)
        counts.insert(0, "task", task_name)
        kplus_tables.append(counts)

kplus_distribution = pd.concat(kplus_tables, ignore_index=True)
kplus_distribution


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(
    data=kplus_distribution[kplus_distribution.label == "non_unimodal"],
    x="threshold",
    y="count",
    hue="task",
    ax=ax,
)
ax.set_title("Number of K+1 examples by nDFU threshold")
ax.set_xlabel("nDFU threshold")
ax.set_ylabel("items assigned to K+1")
sns.despine()


## 11. Save item-level summaries

The grouped item-level summaries are useful outside the notebook. The cell below writes one CSV per task with scalar columns and stringified score/histogram lists.


In [ ]:
out_dir = Path("popquorn_ndfu_outputs")
out_dir.mkdir(exist_ok=True)

for task_name, summary in summaries.items():
    export = summary.copy()
    export["hist"] = export["hist"].apply(lambda x: list(map(float, x)))
    export.to_csv(out_dir / f"{task_name}_item_ndfu.csv", index=False)

sorted(out_dir.glob("*.csv"))


## 12. Notes and next steps

Useful extensions:

- Train a text classifier using the K+1 targets constructed above.
- Compare K+1 thresholds on held-out examples, as in the main nDFU application notebook.
- Replace majority labels with richer targets, such as full distribution prediction.
- Run the subgroup inspection with stricter support thresholds to reduce noisy cases.
- Use the email rewriting task with a separate analysis pipeline for free-form revisions.
